# 02 - Run baseline

Pre: run `01_setup.ipynb` first (needs `clarity/` cloned and installed).

1. Download (eval) data
Checks the md5 against Zenodo

In [3]:
import hashlib, subprocess

URL = 'https://zenodo.org/api/records/14335552/files/cadenza_cad2_task2_eval.v1_1.tar.gz/content'
MD5 = '54268aa8dc08035cd012483bba701d09'
FILE = 'cadenza_cad2_task2_eval.tar.gz'

subprocess.run(['wget', '-q', '-O', FILE, URL], check=True)
md5 = hashlib.md5(open(FILE, 'rb').read()).hexdigest()
assert md5 == MD5, f'md5 mismatch: {md5}'
print('download good')

download good


2. Extract stuff

In [4]:
!tar -xzf cadenza_cad2_task2_eval.tar.gz
!ls cadenza_data/cad2/task2

audio  manifest  metadata  metadata_one_scene


3. Select scene
scene = music clip + target gains (per instrument). Each scene is paired with listeners (audiograms).

In [5]:
import json
from pathlib import Path

root = Path('cadenza_data/cad2/task2').resolve()
meta = root / 'metadata'

scenes = json.load(open(meta / 'scenes.eval.json'))
scene_listeners = json.load(open(meta / 'scene_listeners.eval.json'))
music = json.load(open(meta / 'music.eval.json'))
gains = json.load(open(meta / 'gains.json'))
listeners = json.load(open(meta / 'listeners.eval.json'))

scene_id = 'S62001'
listener_id = scene_listeners[scene_id][0]
scene = scenes[scene_id]

print('scene:', scene_id, scene)
print('listener:', listener_id)
print('gains:', gains[scene['gain']])
print('audiogram levels L:', listeners[listener_id]['audiogram_levels_l'])
print('audiogram levels R:', listeners[listener_id]['audiogram_levels_r'])

scene: S62001 {'music': '17_Nocturne_vn_fl_cl_0001', 'gain': 'gain_0087'}
listener: L6201
gains: {'source_1': 3, 'source_2': 0, 'source_3': -10}
audiogram levels L: [25, 50, 60, 60, 60, 75, 80, 80]
audiogram levels R: [40, 60, 60, 70, 70, 70, 70, 80]


4. Load the audio. Scene uses a short segment of a mixture file which is the full song

In [6]:
import soundfile as sf
from IPython.display import Audio

clip = music[scene['music']]['source_1']
song = clip['track'].split('/')[:3]            # eval/URMP/<song>
song_dir = root / 'audio' / Path(*song)
mix_path = next(song_dir.glob('mix_*.flac'))

audio, sr = sf.read(mix_path)
start, dur = int(clip['start'] * sr), int(clip['duration'] * sr)
segment = audio[start:start + dur]

print('file:', mix_path.name)
print('rate:', sr, 'Hz')
print('full:', audio.shape, f'{len(audio) / sr:.1f}s')
print('segment:', segment.shape, f'{len(segment) / sr:.1f}s')
Audio(segment.T, rate=sr)

file: mix_17_Nocturne_vn_fl_cl.flac
rate: 44100 Hz
full: (4217234, 2) 95.6s
segment: (661500, 2) 15.0s


5. Run baseline on this scene

In [7]:
one = root / 'metadata_one_scene'
one.mkdir(exist_ok=True)
json.dump({scene_id: scene}, open(one / 'scenes.json', 'w'))
json.dump({scene_id: [listener_id]}, open(one / 'scene_listeners.json', 'w'))
print('wrote', sorted(p.name for p in one.iterdir()))

wrote ['scene_listeners.json', 'scenes.json']


In [8]:
exp = Path('exp_one_scene').resolve()
clarity_dir = Path('clarity').resolve()

!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/baseline/enhance.py     path.root={root}     path.metadata_dir={meta}     path.scenes_file={one}/scenes.json     path.scene_listeners_file={one}/scene_listeners.json     path.music_file={meta}/music.eval.json     path.listeners_file={meta}/listeners.eval.json     path.enhancer_params_file={meta}/compressor_params.eval.json     path.exp_folder={exp}

[2026-11-01 02:49:08,849][__main__][INFO] - Loading model cadenzachallenge/ConvTasNet_Bassoon_NonCausal
[2026-11-01 02:49:09,117][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/cadenzachallenge/ConvTasNet_Bassoon_NonCausal/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-11-01 02:49:09,315][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cadenzachallenge/ConvTasNet_Bassoon_NonCausal/5828031665ea5ce9d8b05ec71e53c7eb68fe0761/config.json?%2Fcadenzachallenge%2FConvTasNet_Bassoon_NonCausal%2Fresolve%2Fmain%2Fconfig.json=&etag=%22a049a1a444f751a3571e04e1d01697c68e99788e%22 "HTTP/1.1 200 OK"
[2026-11-01 02:49:09,579][httpx][INFO] - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/cadenzachallenge/ConvTasNet_Bassoon_NonCausal/5828031665ea5ce9d8b05ec71e53c7eb68fe0761/config.json?%2Fcadenzachallenge%2FConvTasNet_Bassoon_NonCausal%2Fresolve%2Fmain%2Fconfig.json=&etag=%22a049a1a444f751a3571e04e1d01697c68e99788e%22 "HTTP/1

6. See the output. Enhanced signal as stereo file.

In [9]:
outs = sorted(exp.rglob('*.flac'))
print(len(outs), 'enhanced file(s)')
for f in outs:
    y, out_sr = sf.read(f)
    print(f.name, y.shape, out_sr, 'Hz')

Audio(y.T, rate=out_sr)

1 enhanced file(s)
S62001_L6201_remix.flac (480000, 2) 32000 Hz


## NEXT
`03_haaqi.ipynb` (HAAQI score).